# Backlog v1 — H04 / H09 / H11 controlled pilots

Frozen attempt-002 weights plus fresh adapters, four prespecified short training
arms, and frozen pooled answer features. Uses fixed IDs and offline pinned inputs.
See docs/SPAN_001_PROTOCOL.md. A complete kernel can contain explicitly recorded
failed/skipped stages; inspect every stage artifact before resolving a task.


In [ ]:
import os,sys,subprocess,time,json,base64,zlib,hashlib
from pathlib import Path
SESSION_STARTED=time.time()
DEADLINE=SESSION_STARTED+25200
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['OMP_NUM_THREADS']='2'
os.environ['OPENBLAS_NUM_THREADS']='2'
os.environ['HF_HUB_OFFLINE']='1'
os.environ['TRANSFORMERS_OFFLINE']='1'
WORK=Path('/kaggle/working')
parents=[Path('/kaggle/input/llm-preference-qwen3-4b-train-002'),Path('/kaggle/input/notebooks/arseniytrushin/llm-preference-qwen3-4b-train-002')]
TRAIN=next(p for p in parents if (p/'joint_run/adapter/adapter_model.safetensors').exists())
ADAPTER=TRAIN/'joint_run/adapter'
assert hashlib.sha256((ADAPTER/'adapter_model.safetensors').read_bytes()).hexdigest()=='8afc09d728698c7bcd02e47f9d427166b4b07b3644c848052cc66c9b4f08e39d'
runtime=json.loads((TRAIN/'runtime_requirements.json').read_text())
subprocess.run([sys.executable,'-m','pip','install','-q','--no-index','--no-deps',
    '--find-links',str(TRAIN/'runtime_wheels'),*runtime],check=True)
MODULES=WORK/'bundled_code'
(MODULES/'arena').mkdir(parents=True,exist_ok=True)
(MODULES/'arena/__init__.py').write_text('')
(WORK/'span_config.json').write_text('{"span": 1, "seed": 20261118, "max_length": 1024, "reward_model": "Skywork/Skywork-Reward-V2-Qwen3-4B", "reward_revision": "fd958fef475f323f4e6b195930e3dd918485c668", "reward_max_session_seconds": 10800, "training_max_session_seconds": 25200, "training_steps": 64, "training_max_seconds_per_arm": 2700, "learning_rate": 2e-05, "weight_decay": 0.01, "warmup_steps": 4, "gradient_accumulation_steps": 8, "random_train_batch_size": 2, "paired_train_batch_size": 1, "eval_batch_size": 4, "attention_rank": 16, "all_linear_attention_rank": 5, "all_linear_mlp_rank": 6, "lora_dropout": 0.05, "consistency_weight": 0.1, "head_pca_components": 16, "head_regularization_grid": [0.01, 0.1, 1.0], "head_inner_group_folds": 3, "calibration_group_folds": 5, "temperature_bounds": [0.5, 3.0], "bootstrap_group_replicates": 2000, "pilot_useful_gain": 0.005, "arms": ["random_attention", "random_all_linear", "paired_attention", "paired_consistency"]}')
(WORK/'base_config.json').write_text('{"attempt": "002", "model_id": "Qwen/Qwen3-4B", "model_source": "qwen-lm/qwen-3/transformers/4b/1", "max_length": 1024, "seed": 20261102, "lora_rank": 16, "lora_alpha": 32, "lora_dropout": 0.05, "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"], "learning_rate": 5e-05, "weight_decay": 0.01, "epochs": 1, "train_batch_size": 4, "eval_batch_size": 4, "gradient_accumulation_steps": 4, "group_training_by_length": true, "eval_steps": 200, "selection_dev_rows": 1024, "max_train_seconds": 21600, "preprocessing_sha256": "a2e047f0e732dd4b1da19fb18baf49ecda888e7602ba3fb353a39c327a93e672", "split_ids_sha256": "066f7ab82a3f14464ce0ef67bfd8b8b34ed190679bfea5c1fea80abc3aa4ecd1"}')
(WORK/'sample_ids.json').write_bytes(zlib.decompress(base64.b64decode('...')))


In [ ]:
(MODULES/'arena/data.py').write_text('from __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport re\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import StratifiedGroupKFold\n\nROOT = Path(os.environ.get("ARENA_DATA_DIR", Path.home() / "database/llm_classification_finetuning"))\nCLASSES = ["winner_model_a", "winner_model_b", "winner_tie"]\nTEXT_COLUMNS = ["prompt", "response_a", "response_b"]\nSEED = 20261101\n\n\ndef parse_turns(value):\n    """Kaggle stores JSON arrays in CSV fields; preserve turn boundaries and nulls."""\n    if not isinstance(value, str):\n        return ""\n    try:\n        turns = json.loads(value)\n    except json.JSONDecodeError as exc:\n        raise ValueError("Expected a JSON-encoded conversation list") from exc\n    if not isinstance(turns, list):\n        raise ValueError("Expected a list of conversation turns")\n    return "\\n\\n[TURN]\\n\\n".join("" if turn is None else str(turn) for turn in turns)\n\n\ndef load_data():\n    train = pd.read_csv(ROOT / "raw/train.csv")\n    test = pd.read_csv(ROOT / "raw/test.csv")\n    for frame in (train, test):\n        for col in TEXT_COLUMNS:\n            frame[col] = frame[col].map(parse_turns)\n    assert train[CLASSES].isin([0, 1]).all().all()\n    assert train[CLASSES].sum(axis=1).eq(1).all()\n    assert train.id.is_unique and test.id.is_unique\n    return train, test\n\n\ndef prompt_groups(frame):\n    return frame.prompt.map(lambda s: hashlib.sha256(re.sub(r"\\s+", " ", s).strip().casefold().encode()).hexdigest())\n\n\ndef split_data(frame):\n    """8 train folds, 1 development fold, 1 untouched final holdout fold."""\n    y = frame[CLASSES].values.argmax(1)\n    groups = prompt_groups(frame)\n    fold = np.full(len(frame), -1)\n    cv = StratifiedGroupKFold(10, shuffle=True, random_state=SEED)\n    for k, (_, idx) in enumerate(cv.split(frame, y, groups)):\n        fold[idx] = k\n    parts = {"train": np.flatnonzero(fold >= 2), "dev": np.flatnonzero(fold == 1), "holdout": np.flatnonzero(fold == 0)}\n    for a, b in [("train", "dev"), ("train", "holdout"), ("dev", "holdout")]:\n        assert not set(groups.iloc[parts[a]]) & set(groups.iloc[parts[b]])\n    return y, parts, fold\n\n\ndef structural_features(frame):\n    """Only features available in test. Never use model identities or IDs."""\n    values = []\n    for col in TEXT_COLUMNS:\n        text = frame[col]\n        n = text.str.len().to_numpy()\n        words = text.str.count(r"\\S+").to_numpy()\n        values.append(np.column_stack([\n            np.log1p(n), np.log1p(words), np.log1p(text.str.count("\\n")),\n            np.log1p(text.str.count("```")), np.log1p(text.str.count(r"[!?]")),\n            np.log1p(text.str.count(r"(?m)^\\s*(?:[-*]|\\d+\\.)\\s")),\n            text.str.contains(r"(?i)\\b(?:sorry|cannot|can\'t|unable|as an ai)\\b", regex=True).astype(float),\n            text.str.count(r"[^\\x00-\\x7F]").to_numpy() / np.maximum(n, 1),\n            (words == 0).astype(float),\n        ]))\n    p, a, b = values\n    return np.column_stack([p, a, b, a-b, np.abs(a-b)]).astype(np.float32)\n\n\ndef swapped(frame):\n    out = frame.copy()\n    out[["response_a", "response_b"]] = frame[["response_b", "response_a"]].values\n    return out\n\n\ndef dense_pair_features(embeddings, stats):\n    p, a, b = embeddings\n    return np.column_stack([p, a, b, a-b, np.abs(a-b), a*b, p*a, p*b, stats]).astype(np.float32)\n\n\ndef swapped_pair_features(embeddings, stats):\n    p, a, b = embeddings\n    d = stats.shape[1] // 5\n    sp, sa, sb, delta, absolute = np.split(stats, [d, 2*d, 3*d, 4*d], axis=1)\n    return dense_pair_features((p, b, a), np.column_stack([sp, sb, sa, -delta, absolute]))\n\n\ndef json_write(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + ".tmp")\n    temp.write_text(json.dumps(value, indent=2, default=lambda x: x.item() if hasattr(x, "item") else str(x)) + "\\n")\n    temp.replace(path)\n')

In [ ]:
(MODULES/'arena/joint_data.py').write_text('"""Token-budgeted joint question/answer inputs and isolated attempt-002 splits."""\nfrom __future__ import annotations\nimport hashlib\nimport json\nimport random\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.model_selection import StratifiedGroupKFold\nfrom torch.utils.data import Dataset\n\nfrom .data import CLASSES, TEXT_COLUMNS, parse_turns, prompt_groups\n\nJOINT_SEED = 20261102\nSWAP_LABELS = np.array([1, 0, 2])\n\n\ndef phase_two_split(frame):\n    labels = frame[CLASSES].values.argmax(1)\n    groups = prompt_groups(frame)\n    folds = np.full(len(frame), -1, dtype=np.int8)\n    splitter = StratifiedGroupKFold(10, shuffle=True, random_state=JOINT_SEED)\n    for k, (_, idx) in enumerate(splitter.split(frame, labels, groups)):\n        folds[idx] = k\n    parts = {"train": np.flatnonzero(folds >= 2), "dev": np.flatnonzero(folds == 0),\n             "confirmation": np.flatnonzero(folds == 1)}\n    for a, b in [("train", "dev"), ("train", "confirmation"), ("dev", "confirmation")]:\n        assert not set(groups.iloc[parts[a]]) & set(groups.iloc[parts[b]])\n    return parts, folds\n\n\ndef split_from_ids(frame, manifest):\n    """Use explicit IDs to keep splits identical across sklearn/runtime versions."""\n    ids = frame.id.to_numpy()\n    parts = {name: np.flatnonzero(np.isin(ids, manifest[name])) for name in ["dev", "confirmation"]}\n    assert len(parts["dev"]) == len(manifest["dev"])\n    assert len(parts["confirmation"]) == len(manifest["confirmation"])\n    assert not set(manifest["dev"]) & set(manifest["confirmation"])\n    excluded = np.concatenate(list(parts.values()))\n    parts["train"] = np.flatnonzero(~np.isin(np.arange(len(frame)), excluded))\n    groups = prompt_groups(frame)\n    for a, b in [("train", "dev"), ("train", "confirmation"), ("dev", "confirmation")]:\n        assert not set(groups.iloc[parts[a]]) & set(groups.iloc[parts[b]])\n    return parts\n\n\ndef read_competition(raw):\n    raw = Path(raw)\n    train, test = pd.read_csv(raw / "train.csv"), pd.read_csv(raw / "test.csv")\n    for frame in (train, test):\n        for col in TEXT_COLUMNS:\n            frame[col] = frame[col].map(parse_turns)\n    return train, test\n\n\ndef head_tail(tokens, budget, marker):\n    if len(tokens) <= budget:\n        return tokens\n    if budget <= len(marker) + 2:\n        return tokens[:budget]\n    remaining = budget - len(marker)\n    first = (remaining + 1) // 2\n    return tokens[:first] + marker + tokens[-(remaining-first):]\n\n\nclass JointTokenizer:\n    """Preserve each field explicitly; truncate fields rather than dropping answer B."""\n    def __init__(self, tokenizer, max_length=1024):\n        self.tokenizer, self.max_length = tokenizer, max_length\n        encode = lambda s: tokenizer.encode(s, add_special_tokens=False)\n        self.prefix = encode("Compare both answers to the user request. Predict the human preference: answer A, answer B, or a tie.\\n\\n[USER REQUEST]\\n")\n        self.a_marker = encode("\\n\\n[ANSWER A]\\n")\n        self.b_marker = encode("\\n\\n[ANSWER B]\\n")\n        self.suffix = encode("\\n\\n[DECISION]\\n")\n        self.truncated = encode("\\n[TRUNCATED]\\n")\n        self.bos = [] if tokenizer.bos_token_id is None else [tokenizer.bos_token_id]\n        self.eos = [] if tokenizer.eos_token_id is None else [tokenizer.eos_token_id]\n        self.overhead = sum(map(len, [self.prefix, self.a_marker, self.b_marker, self.suffix, self.bos, self.eos]))\n        if max_length - self.overhead < 64:\n            raise ValueError("Context budget is too small for all three fields")\n\n    def combine(self, prompt, a, b):\n        available = self.max_length - self.overhead\n        prompt_budget = min(len(prompt), min(256, max(32, available // 4)))\n        answer_budget = available - prompt_budget\n        a_budget, b_budget = min(len(a), answer_budget // 2), min(len(b), answer_budget // 2)\n        unused = answer_budget - a_budget - b_budget\n        take = min(unused, len(a)-a_budget)\n        a_budget += take; unused -= take\n        b_budget += min(unused, len(b)-b_budget)\n        sequence = (self.bos + self.prefix + head_tail(prompt, prompt_budget, self.truncated)\n            + self.a_marker + head_tail(a, a_budget, self.truncated)\n            + self.b_marker + head_tail(b, b_budget, self.truncated) + self.suffix + self.eos)\n        assert len(sequence) <= self.max_length\n        return sequence\n\n    def encode_frame(self, frame):\n        original, swapped = [], []\n        totals = {name: 0 for name in TEXT_COLUMNS}\n        maxima = {name: 0 for name in TEXT_COLUMNS}\n        # Avoid retaining the entire untruncated corpus on each distributed rank.\n        for start in range(0,len(frame),256):\n            fields = []\n            for name in TEXT_COLUMNS:\n                tokens = self.tokenizer(frame[name].iloc[start:start+256].tolist(),\n                    add_special_tokens=False,truncation=False,padding=False)[\'input_ids\']\n                fields.append(tokens)\n                lengths = list(map(len,tokens))\n                totals[name] += sum(lengths)\n                maxima[name] = max(maxima[name],max(lengths,default=0))\n            for p,a,b in zip(*fields):\n                original.append(self.combine(p,a,b))\n                swapped.append(self.combine(p,b,a))\n        diagnostics = {name:{\'mean_tokens\':totals[name]/max(len(frame),1),\n                            \'max_tokens\':maxima[name]} for name in TEXT_COLUMNS}\n        return original, swapped, diagnostics\n\n\nclass PreferenceDataset(Dataset):\n    def __init__(self, original, swapped, labels=None, augment=False, force_swap=False):\n        self.original, self.swapped = original, swapped\n        self.labels, self.augment, self.force_swap = labels, augment, force_swap\n\n    def __len__(self):\n        return len(self.original)\n\n    def __getitem__(self, index):\n        flip = self.force_swap or (self.augment and random.random() < .5)\n        ids = (self.swapped if flip else self.original)[index]\n        item = {"input_ids": ids, "attention_mask": [1]*len(ids)}\n        if self.labels is not None:\n            y = int(self.labels[index])\n            item["labels"] = int(SWAP_LABELS[y]) if flip else y\n        return item\n\n\nclass PreferenceCollator:\n    def __init__(self, tokenizer):\n        self.tokenizer = tokenizer\n\n    def __call__(self, rows):\n        labels = [row["labels"] for row in rows] if "labels" in rows[0] else None\n        inputs = [{k:v for k,v in row.items() if k != "labels"} for row in rows]\n        batch = self.tokenizer.pad(inputs, padding=True, pad_to_multiple_of=8, return_tensors="pt")\n        if labels is not None:\n            batch["labels"] = torch.tensor(labels, dtype=torch.long)\n        return batch\n')

In [ ]:
(MODULES/'arena/joint_train.py').write_text('"""QLoRA joint preference classification, with resumable checkpoints and swap evaluation."""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport json\nimport math\nimport os\nimport platform\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom scipy.optimize import minimize_scalar\nfrom sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, log_loss\nfrom transformers import (AutoModelForSequenceClassification, AutoTokenizer,\n    BitsAndBytesConfig, EarlyStoppingCallback, Trainer, TrainerCallback, TrainingArguments, set_seed)\nfrom peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training\n\nfrom .data import CLASSES, json_write\nfrom .joint_data import (JointTokenizer, PreferenceDataset, PreferenceCollator,\n                         read_competition, split_from_ids)\n\n\ndef probabilities(logits):\n    logits = np.asarray(logits, dtype=np.float64)\n    logits -= logits.max(axis=1, keepdims=True)\n    exp = np.exp(logits)\n    return exp / exp.sum(axis=1, keepdims=True)\n\n\ndef metrics(labels, p):\n    assert p.shape == (len(labels), 3) and np.isfinite(p).all()\n    assert np.allclose(p.sum(1), 1, atol=1e-6)\n    return {"log_loss": float(log_loss(labels, p, labels=[0,1,2])),\n        "accuracy": float(accuracy_score(labels, p.argmax(1))),\n        "balanced_accuracy": float(balanced_accuracy_score(labels, p.argmax(1))),\n        "macro_f1": float(f1_score(labels, p.argmax(1), labels=[0,1,2], average="macro", zero_division=0)),\n        "brier": float(np.mean(np.sum((p-np.eye(3)[labels])**2,axis=1)))}\n\n\ndef temper(p, temperature):\n    return probabilities(np.log(np.clip(p, 1e-12, 1)) / temperature)\n\n\nclass PreferenceTrainer(Trainer):\n    """Mean example loss, with accumulation/DDP scaling owned by Trainer.\n\n    Qwen\'s generic classification forward accepts **kwargs but does not pass\n    num_items_in_batch to its loss. Trainer otherwise assumes that loss was\n    normalized over the accumulated global batch and scales it incorrectly.\n    """\n    def __init__(self, *args, group_training_by_length=False, **kwargs):\n        self.group_training_by_length = group_training_by_length\n        super().__init__(*args, **kwargs)\n        self.model_accepts_loss_kwargs = False\n        self.loss_is_scaled_for_ga = False\n\n    def _get_train_sampler(self, train_dataset=None):\n        if not self.group_training_by_length:\n            return super()._get_train_sampler(train_dataset)\n        from transformers.trainer_pt_utils import LengthGroupedSampler\n        dataset = self.train_dataset if train_dataset is None else train_dataset\n        lengths = [max(len(a),len(b)) for a,b in zip(dataset.original,dataset.swapped)]\n        # Group only training batches. Prediction ordering must stay sequential\n        # so original/swapped probabilities match the external row IDs and labels.\n        return LengthGroupedSampler(self.args.train_batch_size*self.args.gradient_accumulation_steps,\n                                    lengths=lengths)\n\n    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):\n        labels = inputs[\'labels\']\n        outputs = model(**{k: v for k, v in inputs.items() if k != \'labels\'})\n        loss = torch.nn.functional.cross_entropy(outputs.logits.float(), labels)\n        return (loss, outputs) if return_outputs else loss\n\n\nclass Journal(TrainerCallback):\n    def __init__(self, path, max_seconds):\n        self.path, self.max_seconds = Path(path), max_seconds\n        self.started = None\n        self.stopped_for_budget = False\n\n    def on_train_begin(self, args, state, control, **kwargs):\n        self.started = time.monotonic()\n\n    def on_log(self, args, state, control, logs=None, **kwargs):\n        if not state.is_world_process_zero:\n            return\n        record = {"step": state.global_step, "elapsed_seconds": time.monotonic()-self.started, **(logs or {})}\n        with self.path.open("a") as f:\n            f.write(json.dumps(record, default=str)+"\\n")\n        print("JOINT_PROGRESS", json.dumps(record, default=str), flush=True)\n        loss = (logs or {}).get("loss")\n        if loss is not None and not math.isfinite(loss):\n            raise RuntimeError("Non-finite training loss; stopping instead of consuming the run budget")\n\n    def on_step_end(self, args, state, control, **kwargs):\n        stop = bool(self.max_seconds and time.monotonic()-self.started > self.max_seconds)\n        if torch.distributed.is_available() and torch.distributed.is_initialized():\n            flag = torch.tensor(int(stop), device=torch.device("cuda",torch.cuda.current_device()))\n            torch.distributed.all_reduce(flag,op=torch.distributed.ReduceOp.MAX)\n            stop = bool(flag.item())\n        if stop:\n            self.stopped_for_budget = True\n            control.should_training_stop = True\n            control.should_save = True\n            control.should_evaluate = True\n        return control\n\n\ndef load_base_model(model_path, config, training):\n    cuda = torch.cuda.is_available()\n    rank = int(os.environ.get("LOCAL_RANK", "0"))\n    if cuda:\n        torch.cuda.set_device(rank)\n    native_bf16 = cuda and torch.cuda.get_device_capability(rank)[0] >= 8\n    dtype = torch.bfloat16 if native_bf16 else (torch.float16 if cuda else torch.float32)\n    quantize = config.get("quantize", True)\n    if quantize and not cuda:\n        raise RuntimeError("Full QLoRA requires a GPU; CPU is reserved for tiny correctness tests")\n    kwargs = dict(num_labels=3, id2label=dict(enumerate(CLASSES)), label2id={c:i for i,c in enumerate(CLASSES)},\n                  dtype=dtype, attn_implementation="sdpa", local_files_only=True)\n    if quantize:\n        kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",\n            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype,\n            llm_int8_skip_modules=["score"])\n        kwargs["device_map"] = {"": rank}\n    model = AutoModelForSequenceClassification.from_pretrained(str(model_path), **kwargs)\n    model.config.use_cache = False\n    if quantize:\n        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=training,\n            gradient_checkpointing_kwargs={"use_reentrant": False})\n    return model, dtype\n\n\ndef load_trainable_model(model_path, config):\n    model, dtype = load_base_model(model_path, config, training=True)\n    lora = LoraConfig(task_type=TaskType.SEQ_CLS, r=config["lora_rank"],\n        lora_alpha=config["lora_alpha"], lora_dropout=config["lora_dropout"], bias="none",\n        target_modules=config["target_modules"], modules_to_save=["score"])\n    model = get_peft_model(model, lora)\n    assert any("score" in n and p.requires_grad for n,p in model.named_parameters())\n    assert any("lora_" in n and p.requires_grad for n,p in model.named_parameters())\n    return model, dtype\n\n\ndef compute_metrics(prediction):\n    logits = prediction.predictions[0] if isinstance(prediction.predictions, tuple) else prediction.predictions\n    return metrics(prediction.label_ids, probabilities(logits))\n\n\ndef make_trainer(model, tokenizer, train_dataset, dev_dataset, config, out, pilot_steps=0):\n    cuda = torch.cuda.is_available()\n    bf16 = cuda and torch.cuda.get_device_capability()[0] >= 8\n    eval_steps = min(config["eval_steps"], max(1, pilot_steps//2)) if pilot_steps else config["eval_steps"]\n    world = int(os.environ.get("WORLD_SIZE", "1"))\n    total_steps = pilot_steps or math.ceil(config["epochs"]*len(train_dataset)/\n        (config["train_batch_size"]*config["gradient_accumulation_steps"]*world))\n    arguments = TrainingArguments(output_dir=str(out/"checkpoints"),\n        num_train_epochs=config["epochs"], max_steps=pilot_steps if pilot_steps else -1,\n        per_device_train_batch_size=config["train_batch_size"],\n        per_device_eval_batch_size=config["eval_batch_size"],\n        gradient_accumulation_steps=config["gradient_accumulation_steps"],\n        learning_rate=config["learning_rate"], weight_decay=config["weight_decay"],\n        warmup_steps=max(1,int(.05*total_steps)), lr_scheduler_type="cosine", max_grad_norm=1.,\n        gradient_checkpointing=cuda, gradient_checkpointing_kwargs={"use_reentrant":False},\n        bf16=bf16, fp16=cuda and not bf16, eval_strategy="steps", eval_steps=eval_steps,\n        save_strategy="steps", save_steps=eval_steps, save_total_limit=2,\n        load_best_model_at_end=True, metric_for_best_model="log_loss", greater_is_better=False,\n        logging_steps=5, logging_first_step=True, logging_nan_inf_filter=False,\n        report_to="none", seed=config["seed"], data_seed=config["seed"],\n        ddp_find_unused_parameters=False, ddp_timeout=1800, dataloader_num_workers=0, dataloader_pin_memory=cuda,\n        remove_unused_columns=False, optim="adamw_torch", disable_tqdm=True)\n    callbacks = [Journal(out/"training_progress.jsonl", config["max_train_seconds"])]\n    if not pilot_steps:\n        callbacks.append(EarlyStoppingCallback(early_stopping_patience=3))\n    return PreferenceTrainer(model=model, args=arguments, train_dataset=train_dataset, eval_dataset=dev_dataset,\n        group_training_by_length=config.get(\'group_training_by_length\',False),\n        processing_class=tokenizer, data_collator=PreferenceCollator(tokenizer),\n        compute_metrics=compute_metrics, callbacks=callbacks)\n\n\ndef predict_swapped(trainer, original, swapped, labels):\n    def one(force):\n        ds = PreferenceDataset(original, swapped, labels, force_swap=force)\n        logits = trainer.predict(ds).predictions\n        if isinstance(logits, tuple): logits = logits[0]\n        return probabilities(logits)\n    return (one(False)+one(True)[:,[1,0,2]])*.5\n\n\ndef run_training(raw, model_path, split_path, config, out, pilot_steps=0, resume=None):\n    out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    if (out/"summary.json").exists():\n        raise RuntimeError("This training result already exists; use a separate job directory")\n    if config.get(\'preprocessing_sha256\'):\n        assert hashlib.sha256(Path(__file__).with_name(\'joint_data.py\').read_bytes()).hexdigest()==config[\'preprocessing_sha256\']\n    if config.get(\'split_ids_sha256\'):\n        assert hashlib.sha256(Path(split_path).read_bytes()).hexdigest()==config[\'split_ids_sha256\']\n    set_seed(config["seed"])\n    tokenizer=AutoTokenizer.from_pretrained(str(model_path),local_files_only=True,padding_side="right")\n    if tokenizer.pad_token_id is None:\n        tokenizer.pad_token=tokenizer.eos_token\n    frame,_=read_competition(raw)\n    parts=split_from_ids(frame,json.loads(Path(split_path).read_text()))\n    rng=np.random.default_rng(config["seed"])\n    train_idx=parts["train"]\n    if pilot_steps:\n        train_idx=np.sort(rng.choice(train_idx,min(2048,len(train_idx)),replace=False))\n    selection_idx=np.sort(rng.choice(parts["dev"],min(128 if pilot_steps else config["selection_dev_rows"],len(parts["dev"])),replace=False))\n    encoder=JointTokenizer(tokenizer,config["max_length"])\n    train_frame=frame.iloc[train_idx]\n    original,swapped,train_diagnostics=encoder.encode_frame(train_frame)\n    labels=train_frame[CLASSES].values.argmax(1)\n    training=PreferenceDataset(original,swapped,labels,augment=True)\n    dev_frame=frame.iloc[selection_idx]\n    dev_orig,dev_swap,dev_diagnostics=encoder.encode_frame(dev_frame)\n    dev_labels=dev_frame[CLASSES].values.argmax(1)\n    development=PreferenceDataset(dev_orig,dev_swap,dev_labels)\n    model,dtype=load_trainable_model(model_path,config)\n    model.config.pad_token_id=tokenizer.pad_token_id\n    model.config.problem_type="single_label_classification"\n    trainer=make_trainer(model,tokenizer,training,development,config,out,pilot_steps)\n    started=time.monotonic()\n    result=trainer.train(resume_from_checkpoint=resume)\n    train_seconds=time.monotonic()-started\n    trainer.save_model(str(out/"adapter"))\n    if trainer.is_world_process_zero():\n        tokenizer.save_pretrained(out/"adapter")\n        json_write(out/"training_config.json",config)\n    # The confirmation partition is never evaluated during model selection.\n    final_idx=selection_idx if pilot_steps else parts["dev"]\n    val_frame=frame.iloc[final_idx]\n    val_orig,val_swap,_=encoder.encode_frame(val_frame)\n    val_y=val_frame[CLASSES].values.argmax(1)\n    prediction_started=time.monotonic()\n    p=predict_swapped(trainer,val_orig,val_swap,val_y)\n    prediction_seconds=time.monotonic()-prediction_started\n    if trainer.is_world_process_zero():\n        temperature=minimize_scalar(lambda t: metrics(val_y,temper(p,t))["log_loss"],bounds=(.5,3.),method="bounded").x\n        np.savez_compressed(out/"dev_predictions.npz",ids=val_frame.id.to_numpy(),labels=val_y,probabilities=p)\n        world=int(os.environ.get("WORLD_SIZE","1"))\n        summary={"model_id":config["model_id"],"model_path":str(model_path),"pilot":bool(pilot_steps),\n            "config":config,"dtype":str(dtype),"world_size":world,"train_rows":len(train_idx),\n            "selection_dev_rows":len(selection_idx),"final_dev_rows":len(final_idx),\n            "confirmation_evaluated":False,"global_steps":trainer.state.global_step,\n            "epochs_completed":float(trainer.state.epoch),\n            "stopped_for_budget":any(getattr(c,\'stopped_for_budget\',False) for c in trainer.callback_handler.callbacks),\n            "adapter_sha256":hashlib.sha256((out/\'adapter/adapter_model.safetensors\').read_bytes()).hexdigest(),\n            "best_selection_dev_log_loss":trainer.state.best_metric,\n            "best_checkpoint":trainer.state.best_model_checkpoint,\n            "training_seconds":train_seconds,"seconds_per_optimizer_step":train_seconds/max(trainer.state.global_step,1),\n            "development_prediction_seconds":prediction_seconds,\n            "train_result":result.metrics,"development":metrics(val_y,p),\n            "temperature":float(temperature),"calibrated_development":metrics(val_y,temper(p,temperature)),\n            "trainable_parameters":model.get_nb_trainable_parameters()[0],\n            "total_parameters":model.get_nb_trainable_parameters()[1],\n            "peak_gpu_bytes":torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None,\n            "diagnostics":{"train":train_diagnostics,"development":dev_diagnostics},\n            "software":{"python":platform.python_version(),"torch":torch.__version__}}\n        json_write(out/"summary.json",summary)\n        print("JOINT_RESULT",json.dumps(summary),flush=True)\n    trainer.accelerator.wait_for_everyone()\n\n\ndef main():\n    parser=argparse.ArgumentParser()\n    parser.add_argument("--raw",type=Path,required=True)\n    parser.add_argument("--model",type=Path,required=True)\n    parser.add_argument("--split",type=Path,required=True)\n    parser.add_argument("--config",type=Path,required=True)\n    parser.add_argument("--out",type=Path,required=True)\n    parser.add_argument("--pilot-steps",type=int,default=0)\n    parser.add_argument("--resume")\n    args=parser.parse_args()\n    run_training(args.raw,args.model,args.split,json.loads(args.config.read_text()),args.out,args.pilot_steps,args.resume)\n\n\nif __name__=="__main__":\n    main()\n')

In [ ]:
(MODULES/'arena/span_pack.py').write_text('"""Span-specific retained-content helpers; frozen baseline preprocessing stays unchanged."""\nfrom .joint_data import JointTokenizer, head_tail\n\n\nclass SpanTokenizer(JointTokenizer):\n    def retained_parts(self, prompt, a, b):\n        available = self.max_length - self.overhead\n        question = min(len(prompt), min(256, max(32, available // 4)))\n        remaining = available - question\n        a_budget, b_budget = min(len(a), remaining // 2), min(len(b), remaining // 2)\n        unused = remaining - a_budget - b_budget\n        take = min(unused, len(a) - a_budget)\n        a_budget += take\n        unused -= take\n        b_budget += min(unused, len(b) - b_budget)\n        return (head_tail(prompt, question, self.truncated),\n                head_tail(a, a_budget, self.truncated), head_tail(b, b_budget, self.truncated))\n\n    def combine_with_spans(self, prompt, a, b):\n        p, a, b = self.retained_parts(prompt, a, b)\n        first = len(self.bos) + len(self.prefix) + len(p) + len(self.a_marker)\n        second = first + len(a) + len(self.b_marker)\n        ids = self.bos + self.prefix + p + self.a_marker + a + self.b_marker + b + self.suffix + self.eos\n        assert len(ids) <= self.max_length\n        return ids, (first, first + len(a)), (second, second + len(b))\n\n    def reward_pair(self, prompt, a, b):\n        """Use common retained content from both orientations (at most one token shorter)."""\n        p, first, second = self.retained_parts(prompt, a, b)\n        _, swapped_b, swapped_a = self.retained_parts(prompt, b, a)\n        first = head_tail(a, min(len(first), len(swapped_a)), self.truncated)\n        second = head_tail(b, min(len(second), len(swapped_b)), self.truncated)\n        decode = lambda ids: self.tokenizer.decode(ids, skip_special_tokens=False)\n        q, answer_a, answer_b = map(decode, (p, first, second))\n        sequences = []\n        for answer in (answer_a, answer_b):\n            encoded = self.tokenizer.apply_chat_template(\n                [{\'role\': \'user\', \'content\': q}, {\'role\': \'assistant\', \'content\': answer}],\n                tokenize=True, add_generation_prompt=False, return_dict=False)\n            # Transformers versions differ in the default/result container.\n            if hasattr(encoded, \'keys\'):\n                encoded = encoded[\'input_ids\']\n            if hasattr(encoded, \'tolist\'):\n                encoded = encoded.tolist()\n            assert isinstance(encoded, list) and all(isinstance(x, int) for x in encoded), \'Expected flat token IDs\'\n            sequences.append(encoded)\n        assert all(len(ids) <= self.max_length for ids in sequences), \'Reward template exceeds the fixed budget\'\n        return sequences\n')

In [ ]:
(MODULES/'arena/span_adapters.py').write_text('"""Add a trainable pilot adapter without merging or changing the frozen baseline."""\nfrom copy import deepcopy\n\nimport torch\nfrom peft import LoraConfig, TaskType\nfrom peft.tuners.lora.layer import LoraLayer\nfrom peft.utils.other import ModulesToSaveWrapper\n\nATTENTION = [\'q_proj\',\'k_proj\',\'v_proj\',\'o_proj\']\nMLP = [\'gate_proj\',\'up_proj\',\'down_proj\']\n\n\ndef add_span_adapter(model, config, broad=False, frozen=\'frozen\'):\n    raw = model.base_model.model\n    assert isinstance(raw.score, ModulesToSaveWrapper)\n    # Keep the learned head, but only the new adapter should own its trainable copy.\n    raw.score = deepcopy(raw.score.modules_to_save[frozen])\n    rank = config[\'all_linear_mlp_rank\'] if broad else config[\'attention_rank\']\n    rank_pattern = {key: config[\'all_linear_attention_rank\'] for key in ATTENTION} if broad else {}\n    alpha_pattern = {key: 2*value for key, value in rank_pattern.items()}\n    adapter = LoraConfig(task_type=TaskType.SEQ_CLS, r=rank, lora_alpha=2*rank,\n        lora_dropout=config[\'lora_dropout\'], bias=\'none\', target_modules=ATTENTION + (MLP if broad else []),\n        rank_pattern=rank_pattern, alpha_pattern=alpha_pattern, modules_to_save=[\'score\'])\n    model.add_adapter(\'span\', adapter)\n    model.base_model.set_adapter([frozen, \'span\'])\n    for module in model.modules():\n        if isinstance(module, LoraLayer) and frozen in module.lora_dropout:\n            module.lora_dropout[frozen] = torch.nn.Identity()\n    for name, parameter in model.named_parameters():\n        trainable = (\'.lora_A.span.\' in name or \'.lora_B.span.\' in name or\n                     \'.score.modules_to_save.span.\' in name)\n        parameter.requires_grad_(trainable)\n    trainable = {name: p.numel() for name, p in model.named_parameters() if p.requires_grad}\n    assert trainable and all(\'.span.\' in name for name in trainable)\n    assert any(\'.score.\' in name for name in trainable)\n    return {\'trainable_parameters\': sum(trainable.values()),\n            \'lora_parameters\': sum(size for name, size in trainable.items() if \'.lora_\' in name),\n            \'head_parameters\': sum(size for name, size in trainable.items() if \'.score.\' in name),\n            \'target_modules\': sorted(adapter.target_modules), \'rank_pattern\': rank_pattern}\n\n\ndef paired_objective(logits, labels, consistency_weight=0.):\n    """Rows are [all originals, all swapped]; labels must be aligned accordingly."""\n    assert len(logits) % 2 == 0\n    half = len(logits) // 2\n    swap = torch.tensor([1,0,2], device=labels.device)\n    assert torch.equal(labels[half:], swap[labels[:half]])\n    loss = torch.nn.functional.cross_entropy(logits.float(), labels)\n    first = torch.log_softmax(logits[:half].float(), dim=1)\n    second = torch.log_softmax(logits[half:].float()[:,[1,0,2]], dim=1)\n    kl = .5*((first.exp()*(first-second)).sum(1) +\n             (second.exp()*(second-first)).sum(1)).mean()\n    return loss + consistency_weight*kl, kl\n')

In [ ]:
(MODULES/'arena/span_joint.py').write_text('"""Frozen pooled features and controlled warm-start training for H04/H09/H11."""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nimport torch\nfrom peft import PeftModel\nfrom transformers import AutoTokenizer, TrainingArguments, set_seed\n\nfrom .data import CLASSES, TEXT_COLUMNS, json_write\nfrom .joint_data import PreferenceDataset, PreferenceCollator, read_competition\nfrom .joint_train import Journal, PreferenceTrainer, load_base_model, probabilities\nfrom .span_adapters import add_span_adapter, paired_objective, ATTENTION\nfrom .span_pack import SpanTokenizer\n\n\nclass PairDataset(PreferenceDataset):\n    def __getitem__(self, index):\n        return {\'input_ids\': self.original[index], \'swapped_input_ids\': self.swapped[index],\n                \'labels\': int(self.labels[index])}\n\n\nclass PairCollator:\n    def __init__(self, tokenizer):\n        self.single = PreferenceCollator(tokenizer)\n\n    def __call__(self, rows):\n        records = [{\'input_ids\':row[\'input_ids\'],\'attention_mask\':[1]*len(row[\'input_ids\']),\n                    \'labels\':row[\'labels\']} for row in rows]\n        records += [{\'input_ids\':row[\'swapped_input_ids\'],\n                     \'attention_mask\':[1]*len(row[\'swapped_input_ids\']),\n                     \'labels\':[1,0,2][row[\'labels\']]} for row in rows]\n        result = self.single(records)\n        result[\'paired\'] = True\n        return result\n\n\nclass SpanTrainer(PreferenceTrainer):\n    def __init__(self, *args, consistency_weight=0., **kwargs):\n        super().__init__(*args, **kwargs)\n        self.consistency_weight = consistency_weight\n\n    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):\n        labels = inputs[\'labels\']\n        paired = inputs.get(\'paired\', False)\n        outputs = model(**{key:value for key,value in inputs.items() if key not in (\'labels\',\'paired\')})\n        if paired:\n            loss, _ = paired_objective(outputs.logits, labels, self.consistency_weight)\n        else:\n            loss = torch.nn.functional.cross_entropy(outputs.logits.float(), labels)\n        return (loss, outputs) if return_outputs else loss\n\n\ndef load_frozen(model_path, adapter, base_config, training=False):\n    base, dtype = load_base_model(model_path, base_config, training=training)\n    model = PeftModel.from_pretrained(base, str(adapter), adapter_name=\'frozen\', is_trainable=False)\n    model.config.use_cache = False\n    return model, dtype\n\n\ndef select_rows(raw, samples, keys):\n    frame, _ = read_competition(raw)\n    ids = [key for name in keys for key in samples[name]]\n    assert len(ids) == len(set(ids))\n    selected = frame.loc[frame.id.isin(ids)].copy()\n    assert len(selected) == len(ids)\n    return selected\n\n\ndef collect_features(raw, model_path, adapter, samples, config, base_config, out, deadline):\n    rank, world = int(os.environ.get(\'LOCAL_RANK\',\'0\')),int(os.environ.get(\'WORLD_SIZE\',\'1\'))\n    torch.cuda.set_device(rank)\n    selected = select_rows(raw,samples,[\'feature_fit\',\'dev\']).iloc[rank::world]\n    tokenizer = AutoTokenizer.from_pretrained(adapter,local_files_only=True,padding_side=\'right\')\n    if tokenizer.pad_token_id is None: tokenizer.pad_token = tokenizer.eos_token\n    encoder = SpanTokenizer(tokenizer,config[\'max_length\'])\n    views, spans = [[],[]],[[],[]]\n    for row in selected.itertuples():\n        p,a,b = [tokenizer.encode(getattr(row,key),add_special_tokens=False) for key in TEXT_COLUMNS]\n        for index, first, second in [(0,a,b),(1,b,a)]:\n            ids, sa, sb = encoder.combine_with_spans(p,first,second)\n            views[index].append(ids);spans[index].append((sa,sb))\n    model, _ = load_frozen(model_path,adapter,base_config)\n    model.eval();model.config.pad_token_id = tokenizer.pad_token_id\n    raw_model = model.get_base_model()\n    hidden = model.config.hidden_size\n    n = len(selected)\n    pooled_a = np.full((2,n,hidden),np.nan,dtype=np.float16)\n    pooled_b = np.full_like(pooled_a,np.nan)\n    last = np.full_like(pooled_a,np.nan)\n    logits = np.full((2,n,3),np.nan,dtype=np.float32)\n    batch_size = config[\'eval_batch_size\']\n    started = time.monotonic()\n    out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    for view in range(2):\n        order = np.argsort([len(ids) for ids in views[view]],kind=\'stable\')\n        for begin in range(0,n,batch_size):\n            if time.time() > deadline-300:\n                raise TimeoutError(\'Feature extraction reached the shared session ceiling\')\n            chosen = order[begin:begin+batch_size]\n            batch = tokenizer.pad([{\'input_ids\':views[view][i],\n                                    \'attention_mask\':[1]*len(views[view][i])} for i in chosen],\n                                   padding=True,pad_to_multiple_of=8,return_tensors=\'pt\')\n            batch = {key:value.to(rank) for key,value in batch.items()}\n            with torch.inference_mode(),torch.autocast(\'cuda\',dtype=torch.float16):\n                h = raw_model.model(**batch,use_cache=False).last_hidden_state\n                token_logits = raw_model.score(h)\n                for offset,index in enumerate(chosen):\n                    sa,sb = spans[view][index]\n                    a = h[offset,sa[0]:sa[1]].mean(0) if sa[1]>sa[0] else torch.zeros(hidden,device=rank)\n                    b = h[offset,sb[0]:sb[1]].mean(0) if sb[1]>sb[0] else torch.zeros(hidden,device=rank)\n                    pooled_a[view,index] = a.float().cpu().numpy()\n                    pooled_b[view,index] = b.float().cpu().numpy()\n                    last[view,index] = h[offset,len(views[view][index])-1].float().cpu().numpy()\n                    logits[view,index] = token_logits[offset,len(views[view][index])-1].float().cpu().numpy()\n            if begin % 256 < batch_size:\n                print(\'SPAN_FEATURE_PROGRESS\',json.dumps({\'rank\':rank,\'view\':view,\'rows_in_view\':int(begin+len(chosen)),\n                     \'total_rows\':n,\'elapsed_seconds\':time.monotonic()-started}),flush=True)\n    original, swapped = probabilities(logits[0]),probabilities(logits[1])[:,[1,0,2]]\n    values = {\'ids\':selected.id.to_numpy(), \'answer_a\':(pooled_a[0].astype(float)+pooled_b[1])/2,\n              \'answer_b\':(pooled_b[0].astype(float)+pooled_a[1])/2,\n              \'joint\':(last[0].astype(float)+last[1])/2,\n              \'probabilities\':(original+swapped)/2,\'original\':original,\'aligned_swap\':swapped,\n              \'feature_fit\':selected.id.isin(samples[\'feature_fit\']).to_numpy()}\n    assert all(np.isfinite(value).all() for key,value in values.items() if key != \'feature_fit\')\n    for key in (\'answer_a\',\'answer_b\',\'joint\'): values[key] = values[key].astype(np.float16)\n    np.savez_compressed(out/f\'features_rank{rank}.npz\',**values)\n    json_write(out/f\'features_rank{rank}.json\',{\'rank\':rank,\'rows\':n,\'complete\':True,\n        \'seconds\':time.monotonic()-started,\'peak_gpu_bytes\':torch.cuda.max_memory_allocated(rank)})\n\n\ndef run_training(raw, model_path, adapter, samples, config, base_config, out, deadline, arm):\n    set_seed(config[\'seed\'])\n    out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    assert not (out/\'summary.json\').exists(), \'Never overwrite a completed arm\'\n    selected = select_rows(raw,samples,[\'training_pilot\'])\n    development = select_rows(raw,samples,[\'dev\'])\n    tokenizer=AutoTokenizer.from_pretrained(adapter,local_files_only=True,padding_side=\'right\')\n    if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token\n    encoder=SpanTokenizer(tokenizer,config[\'max_length\'])\n    original,swapped,train_diagnostics=encoder.encode_frame(selected)\n    labels=selected[CLASSES].values.argmax(1)\n    paired=arm.startswith(\'paired\')\n    training=(PairDataset(original,swapped,labels) if paired else\n              PreferenceDataset(original,swapped,labels,augment=True))\n    dev_orig,dev_swap,dev_diagnostics=encoder.encode_frame(development)\n    dev_y=development[CLASSES].values.argmax(1)\n    model,dtype=load_frozen(model_path,adapter,base_config,training=True)\n    model.config.pad_token_id=tokenizer.pad_token_id\n    model.config.problem_type=\'single_label_classification\'\n    with torch.no_grad():\n        example=tokenizer.pad([{\'input_ids\':dev_orig[0],\'attention_mask\':[1]*len(dev_orig[0])}],\n                               return_tensors=\'pt\')\n        device=torch.cuda.current_device()\n        example={key:value.to(device) for key,value in example.items()}\n        model.eval()\n        with torch.autocast(\'cuda\',dtype=torch.float16): before=model(**example).logits.detach().clone()\n    counts=add_span_adapter(model,config,broad=(arm==\'random_all_linear\'))\n    reference=sum((module.in_features+module.out_features)*config[\'attention_rank\']\n                  for name,module in model.named_modules()\n                  if name.split(\'.\')[-1] in ATTENTION and hasattr(module,\'lora_A\'))\n    counts[\'reference_attention_lora_parameters\']=reference\n    counts[\'relative_lora_count_change\']=(counts[\'lora_parameters\']-reference)/reference\n    assert abs(counts[\'relative_lora_count_change\']) < .02\n    model.eval()\n    with torch.no_grad(),torch.autocast(\'cuda\',dtype=torch.float16): after=model(**example).logits\n    assert torch.allclose(before,after,atol=2e-3,rtol=2e-3), \'Adding zero-initialized adapters changed baseline outputs\'\n    # Module initialization consumes different RNG amounts for broad coverage;\n    # reset before sampling so the two random-view arms use the same row order.\n    set_seed(config[\'seed\'])\n    batch=config[\'paired_train_batch_size\' if paired else \'random_train_batch_size\']\n    training_cap=min(config[\'training_max_seconds_per_arm\'],max(0,deadline-time.time()-900))\n    assert training_cap >= 300, \'Insufficient remaining budget for this training/evaluation arm\'\n    arguments=TrainingArguments(output_dir=str(out/\'checkpoints\'),max_steps=config[\'training_steps\'],\n        per_device_train_batch_size=batch,per_device_eval_batch_size=config[\'eval_batch_size\'],\n        gradient_accumulation_steps=config[\'gradient_accumulation_steps\'],\n        learning_rate=config[\'learning_rate\'],weight_decay=config[\'weight_decay\'],\n        warmup_steps=config[\'warmup_steps\'],lr_scheduler_type=\'cosine\',max_grad_norm=1.,\n        gradient_checkpointing=True,gradient_checkpointing_kwargs={\'use_reentrant\':False},\n        fp16=True,bf16=False,eval_strategy=\'no\',save_strategy=\'no\',report_to=\'none\',\n        logging_steps=8,logging_first_step=True,logging_nan_inf_filter=False,\n        include_num_input_tokens_seen=\'non_padding\',\n        seed=config[\'seed\'],data_seed=config[\'seed\'],ddp_find_unused_parameters=False,\n        dataloader_num_workers=0,remove_unused_columns=False,disable_tqdm=True,optim=\'adamw_torch\')\n    journal=Journal(out/\'training_progress.jsonl\',training_cap)\n    trainer=SpanTrainer(model=model,args=arguments,train_dataset=training,\n        processing_class=tokenizer,data_collator=PairCollator(tokenizer) if paired else PreferenceCollator(tokenizer),\n        callbacks=[journal],group_training_by_length=True,\n        consistency_weight=config[\'consistency_weight\'] if arm==\'paired_consistency\' else 0.)\n    started=time.monotonic()\n    result=trainer.train()\n    train_seconds=time.monotonic()-started\n    model.save_pretrained(out/\'adapter\',selected_adapters=[\'span\']) if trainer.is_world_process_zero() else None\n    trainer.accelerator.wait_for_everyone()\n    # Use the ordinary evaluation collator even for paired-view training.\n    trainer.data_collator=PreferenceCollator(tokenizer)\n    outputs=[]\n    for force in (False,True):\n        ds=PreferenceDataset(dev_orig,dev_swap,dev_y,force_swap=force)\n        values=trainer.predict(ds).predictions\n        if isinstance(values,tuple): values=values[0]\n        outputs.append(probabilities(values))\n    if trainer.is_world_process_zero():\n        np.savez_compressed(out/\'predictions.npz\',ids=development.id.to_numpy(),labels=dev_y,\n            original=outputs[0],aligned_swap=outputs[1][:,[1,0,2]],\n            probabilities=(outputs[0]+outputs[1][:,[1,0,2]])/2)\n        json_write(out/\'summary.json\',{\'arm\':arm,\'complete\':True,\'steps\':trainer.state.global_step,\n            \'stopped_for_budget\':journal.stopped_for_budget,\'training_seconds\':train_seconds,\n            \'canonical_rows\':len(selected),\'development_rows\':len(development),\'world_size\':2,\n            \'views_per_update\':batch*config[\'gradient_accumulation_steps\']*2*(2 if paired else 1),\n            \'canonical_exposures\':trainer.state.global_step*batch*config[\'gradient_accumulation_steps\']*2,\n            \'non_padding_tokens_seen\':int(trainer.state.num_input_tokens_seen),\n            \'initial_logit_max_difference\':float((before-after).abs().max()),\'counts\':counts,\n            \'peak_gpu_bytes\':torch.cuda.max_memory_allocated(),\'train_result\':result.metrics,\n            \'train_diagnostics\':train_diagnostics,\'dev_diagnostics\':dev_diagnostics,\'config\':config})\n    trainer.accelerator.wait_for_everyone()\n\n\ndef main():\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'mode\',choices=[\'features\',\'train\'])\n    for name in (\'raw\',\'model\',\'adapter\',\'samples\',\'config\',\'base-config\',\'out\'):\n        parser.add_argument(\'--\'+name,type=Path,required=True)\n    parser.add_argument(\'--deadline\',type=float,required=True)\n    parser.add_argument(\'--arm\')\n    args=parser.parse_args()\n    samples=json.loads(args.samples.read_text());config=json.loads(args.config.read_text())\n    base_config=json.loads(args.base_config.read_text())\n    kwargs=dict(raw=args.raw,model_path=args.model,adapter=args.adapter,samples=samples,\n                config=config,base_config=base_config,out=args.out,deadline=args.deadline)\n    if args.mode==\'features\':collect_features(**kwargs)\n    else:\n        assert args.arm in config[\'arms\']\n        run_training(**kwargs,arm=args.arm)\n\n\nif __name__==\'__main__\':main()\n')

In [ ]:
import torch
assert torch.cuda.device_count()==2
print('GPUS',[(torch.cuda.get_device_name(i),torch.cuda.get_device_properties(i).total_memory) for i in range(2)])
assert hashlib.sha256((MODULES/'arena/joint_data.py').read_bytes()).hexdigest()=='a2e047f0e732dd4b1da19fb18baf49ecda888e7602ba3fb353a39c327a93e672'
RAW=next(p for p in [Path('/kaggle/input/llm-classification-finetuning'),
    Path('/kaggle/input/competitions/llm-classification-finetuning')] if (p/'train.csv').exists())
MODEL=next(p for p in [Path('/kaggle/input/qwen-3/transformers/4b/1'),
    Path('/kaggle/input/models/qwen-lm/qwen-3/transformers/4b/1'),TRAIN/'base_model'] if (p/'config.json').exists())
environment=os.environ.copy();environment['PYTHONPATH']=str(MODULES);environment['PYTHONUNBUFFERED']='1'
stages=[]
def execute(mode,name,stage_deadline,arm=None):
    directory=WORK/'joint'/name
    directory.mkdir(parents=True,exist_ok=True)
    command=[sys.executable,'-m','torch.distributed.run','--standalone','--nproc_per_node=2',
        '--module','arena.span_joint',mode,'--raw',str(RAW),'--model',str(MODEL),
        '--adapter',str(ADAPTER),'--samples',str(WORK/'sample_ids.json'),
        '--config',str(WORK/'span_config.json'),'--base-config',str(WORK/'base_config.json'),
        '--out',str(directory),'--deadline',str(stage_deadline)]
    if arm:command+=['--arm',arm]
    started=time.time()
    with (directory/'console.log').open('w') as log:
        process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
            text=True,bufsize=1,env=environment)
        for line in process.stdout:
            print(line,end='',flush=True);log.write(line);log.flush()
        status=process.wait()
    result={'stage':name,'exit_code':status,'elapsed_seconds':time.time()-started,'complete':status==0}
    stages.append(result)
    (directory/'stage_status.json').write_text(json.dumps(result,indent=2))
    print('SPAN_STAGE',json.dumps(result),flush=True)
    return status==0
feature_deadline=min(DEADLINE-4*3600,time.time()+5400)
execute('features','features',feature_deadline)
for arm in ['random_attention', 'random_all_linear', 'paired_attention', 'paired_consistency']:
    if time.time()>DEADLINE-1200:
        stages.append({'stage':arm,'complete':False,'reason':'shared session budget exhausted'})
        continue
    execute('train',arm,DEADLINE,arm=arm)
summary={'stages':stages,'all_complete':all(s['complete'] for s in stages),
         'session_seconds':time.time()-SESSION_STARTED,'base_adapter_sha256':'8afc09d728698c7bcd02e47f9d427166b4b07b3644c848052cc66c9b4f08e39d'}
(WORK/'joint_summary.json').write_text(json.dumps(summary,indent=2))
if not any(s['complete'] for s in stages):raise RuntimeError('No pilot stage completed; inspect retained logs')
summary
